# 8-bit attention (SageAttention2) on top of compile: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_sage_2026-11-11.md` (S1–S8). All runs share this session.

Time: about 10 min to build SageAttention, then about 75 min of runs (~3 compute units). The last section (fp16-PV variant) is optional.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)
!nvcc --version | tail -2

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10 min)

Needs nvcc ≥ 12.4 with the same major version as torch's CUDA (printed above). If the build fails, send me the last lines of its output.

In [ ]:
%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation . 2>&1 | tail -15
%cd /content/video-dit-l4
!python -c "import importlib.metadata as m, sageattention; print('sageattention', m.version('sageattention'))"

## Kernel microbenchmark (S2, ~1 min)

In [ ]:
!python bench/attn_micro.py --out results/sage_micro.json

## 33 frames, 4 prompts

Compile with flash attention: this session's baseline (~11 min). The hash check is S1; it only applies if torch is still 2.14.1.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --kernels --out results/compile_33f_s3
!python bench/hash_frames.py results/compile_33f_s3 --check results/compile_rerun_33f/frames_sha256.json

Compile with SageAttention (~9 min). If compiling fails with a graph-break error, add `--no-fullgraph` and note it.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend sage --kernels --out results/sage_33f

In [ ]:
!python bench/compare_frames.py --ref results/compile_33f_s3 --test results/sage_33f

## fp32 reference, fox and market (~28 min), then S6

The .npy files of this run go into the zip, so later experiments can be compared against fp32 without rerunning it.

In [ ]:
!python bench/profile_baseline.py --frames 33 --dtype fp32 --attn efficient --prompt-ids 0 3 --out results/fp32_33f_s3
!python bench/hash_frames.py results/fp32_33f_s3 --check results/fp32_33f/frames_sha256.json

In [ ]:
for t in ['compile_33f_s3', 'sage_33f']:
    print('==', t)
    !python bench/compare_frames.py --ref results/fp32_33f_s3 --test results/{t} --out-name compare_vs_fp32.json

## 81 frames, fox (~17 min), S3, S4 and S8

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --kernels --out results/compile_81f_s3

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend sage --kernels --out results/sage_81f

In [ ]:
!python bench/compare_frames.py --ref results/compile_81f_s3 --test results/sage_81f

## Optional: INT8 QK with fp16 PV (S7, ~10 min)

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/sage_fp16pv_33f
!python bench/compare_frames.py --ref results/compile_33f_s3 --test results/sage_fp16pv_33f
!python bench/compare_frames.py --ref results/fp32_33f_s3 --test results/sage_fp16pv_33f --out-name compare_vs_fp32.json

In [ ]:
# Every run's hashes; then zip without .npy, except the fp32 reference (~80 MB)
import glob, os
for d in sorted(glob.glob('results/*_s3') + glob.glob('results/sage_*')):
    if os.path.isdir(d) and glob.glob(d + '/frames_*.npy'):
        !python bench/hash_frames.py {d}
!zip -qr sage_results.zip results -x '*.npy' && zip -qr sage_results.zip results/fp32_33f_s3/*.npy && ls -lh sage_results.zip
from google.colab import files; files.download('sage_results.zip')